# HORM E/F versus E/F/H screening

Open this notebook from GitHub and use a fresh GPU runtime. Its first cell uploads the bundle downloaded by stage 01; its final cell downloads a cumulative local bundle for stage 03.

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one stage-01 .tar.gz bundle.'

In [ ]:
from pathlib import Path
import tarfile

uploaded_name = next(iter(uploaded))
input_bundle = Path('/content') / uploaded_name
if not input_bundle.is_file():
    input_bundle.write_bytes(uploaded[uploaded_name])
OUTPUT_ROOT = Path('/content/oa_audit_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_root_resolved = OUTPUT_ROOT.resolve()
with tarfile.open(input_bundle, 'r:gz') as archive:
    for member in archive.getmembers():
        target = (OUTPUT_ROOT / member.name).resolve()
        assert target == output_root_resolved or output_root_resolved in target.parents
        assert member.isfile() or member.isdir(), f'Unsupported archive entry: {member.name}'
    archive.extractall(OUTPUT_ROOT)

REPOSITORY_URL = 'https://github.com/jiaxi98/OAReactDiff.git'
REPOSITORY_REF = 'agent/oa-failure-audit'
REPO = Path('/content/OAReactDiff')
MANIFEST = OUTPUT_ROOT / 'generation_8x8_r2_j2/candidate_manifest.csv'
SCREEN_ROOT = OUTPUT_ROOT / 'horm_screen_generation_8x8_r2_j2'
MODEL_ROOT = Path('/content/models/HORM')
assert MANIFEST.is_file(), f'The uploaded stage-01 bundle is missing {MANIFEST}'
SCREEN_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
if not (REPO / '.git').is_dir():
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPOSITORY_REF} {REPOSITORY_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin {REPOSITORY_REF}
    !git -C {REPO} checkout --detach FETCH_HEAD
!git -C {REPO} rev-parse HEAD
!cd {REPO} && bash experiments/oa_failure_audit/setup_colab_horm.sh

In [ ]:
REVISION = 'a582ca8f0bfb9c50c6e384e4ef1db9b0a8dd1dd4'
EF = MODEL_ROOT / 'left_orig.ckpt'
EFH = MODEL_ROOT / 'left.ckpt'
if not EF.is_file():
    !curl -L --fail --output {EF} https://huggingface.co/yhong55/HORM/resolve/{REVISION}/left_orig.ckpt
if not EFH.is_file():
    !curl -L --fail --output {EFH} https://huggingface.co/yhong55/HORM/resolve/{REVISION}/left.ckpt
!echo '1c286d36152781d1923cf6ab778d2f5227cf8bb626e07604dc8b86f15a6ac6fa  '{EF} | sha256sum --check
!echo '55b1f2d21897ad4f7870986397ed981185989fc947f08aff172c65cd41a1f2a0  '{EFH} | sha256sum --check

In [ ]:
MAMBA = '/usr/local/bin/micromamba'
ENV_PREFIX = '/content/micromamba/envs/oa-horm'
HORM_REPO = Path('/content/HORM')
EF_RESULT = SCREEN_ROOT / 'horm_left_ef.csv'
EFH_RESULT = SCREEN_ROOT / 'horm_left_efh.csv'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/screen_horm.py \
    --manifest {MANIFEST} --horm-repo {HORM_REPO} --checkpoint {EF} \
    --label horm_left_ef --output {EF_RESULT} --device cuda --resume

In [ ]:
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/screen_horm.py \
    --manifest {MANIFEST} --horm-repo {HORM_REPO} --checkpoint {EFH} \
    --label horm_left_efh --output {EFH_RESULT} --device cuda --resume

In [ ]:
ENRICHED = SCREEN_ROOT / 'candidate_manifest_screened.csv'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/merge_screening.py \
    --manifest {MANIFEST} --screen horm_left_ef={EF_RESULT} \
    --screen horm_left_efh={EFH_RESULT} --output {ENRICHED} --overwrite
DFT_SUBSET = SCREEN_ROOT / 'dft_subset.csv'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/select_dft_subset.py \
    --manifest {ENRICHED} --output {DFT_SUBSET} --budget 96 --max-per-reaction 2 --overwrite

The 8 × 8 pilot can contribute at most 16 DFT cases under the two-per-reaction cap. Generate at least 25–50 reactions before constructing the intended 50–200 case DFT subset.

## Download the screening results to this computer

This cell packages the stage-01 structures, HORM results, and DFT worklist into one cumulative browser download. Keep it for stage 03 and local analysis.

In [ ]:
import hashlib
import shutil
from google.colab import files

bundle_path = Path(shutil.make_archive(
    '/content/oa_audit_02_horm_screen_generation_8x8_r2_j2',
    'gztar',
    root_dir=OUTPUT_ROOT,
    base_dir='.',
))
digest = hashlib.sha256()
with bundle_path.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
print(f'{bundle_path.name}: {bundle_path.stat().st_size / 1024**2:.2f} MiB')
print(f'sha256: {digest.hexdigest()}')
files.download(str(bundle_path))